# Neo4j GraphRAG With llama-index-pydocker

GraphRAG retrieval outperforms dense-vector search on multi-hop questions because it can follow relationships across the graph rather than returning the nearest embedding. Neo4j is the most widely used graph database for this pattern.

This notebook shows how `llama-index-pydocker` replaces the upstream `Neo4jGraphStore` with a single import swap. On a localhost URL the container is provisioned automatically; on a cloud URL the wrapper is invisible.

## Table Of Contents

- [Prerequisites](#prerequisites)
- [1. Start Neo4j With One Import Swap](#1-start-neo4j-with-one-import-swap)
- [2. Write And Query The Knowledge Graph](#2-write-and-query-the-knowledge-graph)
- [3. Index Documents With LlamaIndex](#3-index-documents-with-llamaindex)
- [4. Context Manager Teardown](#4-context-manager-teardown)
- [5. Remote Passthrough (No Docker)](#5-remote-passthrough-no-docker)
- [6. Conclusion](#6-conclusion)

## Prerequisites

- Docker Desktop must be running.
- Install dependencies:
  ```bash
  pip install "llama-index-pydocker[neo4j]"
  ```
- No environment variables required — credentials are set in the config below.

## 1. Start Neo4j With One Import Swap

Change the import from `llama_index.graph_stores.neo4j` to `llama_index_pydocker`. Everything else — constructor signature, method names, LlamaIndex integration — stays identical.

In [ ]:
import sys
import uuid
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent))

# One import swap — Docker container starts automatically for localhost URLs
from llama_index_pydocker import Neo4jGraphStore
from docker_db import Neo4jConfig

In [ ]:
temp_dir = Path(tempfile.mkdtemp())
container_name = f"demo-neo4j-{uuid.uuid4().hex[:8]}"

cfg = Neo4jConfig(
    password="demopassword",
    project_name="demo",
    container_name=container_name,
    volume_path=temp_dir / "neo4jdata",
    retries=40,
    delay=3,
)

store = Neo4jGraphStore(
    url=f"bolt://localhost:{cfg.port}",
    password="demopassword",
    docker_config=cfg,
)

print(f"Neo4j started: {container_name}")
print(f"Container:     {store._db.config.container_name}")

## 2. Write And Query The Knowledge Graph

`Neo4jGraphStore` exposes the underlying `neo4j.Driver` through its `_driver` attribute. Use it to run Cypher directly.

In [ ]:
driver = store._driver

with driver.session(database="neo4j") as session:
    session.run(
        "CREATE (p:Paper {title: $title, year: $year})",
        title="Attention Is All You Need", year=2017,
    )
    session.run(
        "CREATE (r:Researcher {name: $name})",
        name="Vaswani",
    )
    session.run(
        """
        MATCH (p:Paper {title: 'Attention Is All You Need'}),
              (r:Researcher {name: 'Vaswani'})
        CREATE (p)-[:AUTHORED_BY]->(r)
        """
    )

    result = session.run(
        "MATCH (p:Paper)-[:AUTHORED_BY]->(r) RETURN p.title AS title, r.name AS author"
    )
    for rec in result:
        print(rec["title"], "→", rec["author"])

## 3. Index Documents With LlamaIndex

Pass `store` directly to `StorageContext`. LlamaIndex writes triplets to Neo4j through the same connection the wrapper manages.

In [ ]:
from llama_index.core import StorageContext, KnowledgeGraphIndex
from llama_index.core.schema import Document
from llama_index.core.embeddings import MockEmbedding

documents = [
    Document(text="Vaswani introduced the Transformer architecture in 2017."),
    Document(text="The Transformer replaced recurrent networks for sequence modelling."),
]

storage_context = StorageContext.from_defaults(graph_store=store)

index = KnowledgeGraphIndex.from_documents(
    documents,
    storage_context=storage_context,
    max_triplets_per_chunk=3,
    include_embeddings=False,
)

query_engine = index.as_query_engine(
    include_text=True,
    retriever_mode="keyword",
)

response = query_engine.query("What did Vaswani introduce?")
print(response)

## 4. Context Manager Teardown

Call `store.stop()` to remove the container, or use `with` for automatic teardown.

In [ ]:
store.stop()
print(f"Container '{container_name}' removed.")

In [ ]:
# Equivalent pattern with automatic teardown
new_name = f"demo-neo4j-{uuid.uuid4().hex[:8]}"
new_cfg = cfg.model_copy(update={"container_name": new_name,
                                   "volume_path": temp_dir / new_name})

with Neo4jGraphStore(
    url=f"bolt://localhost:{new_cfg.port}",
    password="demopassword",
    docker_config=new_cfg,
) as s:
    print(f"Container up: {s._db.config.container_name}")
# Container removed here automatically
print("Container removed.")

## 5. Remote Passthrough (No Docker)

Pass a non-localhost URL and the wrapper becomes invisible — no container is started, no Docker daemon is contacted.

In [ ]:
# Uncomment and fill in real credentials to use a hosted Neo4j AuraDB instance
#
# store = Neo4jGraphStore(
#     url="bolt+ssc://my-aura.databases.neo4j.io:7687",
#     username="neo4j",
#     password="<aura-password>",
# )
# assert store._db is None   # no container

print("Remote passthrough: Docker is never touched for non-localhost URLs.")

## 6. Conclusion

You have completed the workflow introduced at the top:
1. Provisioned a local Neo4j container with a single import swap.
2. Written and queried a knowledge graph using Cypher directly.
3. Built a `KnowledgeGraphIndex` and issued a GraphRAG query.
4. Removed the container with `stop()` and with a context manager.

Key next steps:
- Replace `include_embeddings=False` and add a real embedding model for hybrid retrieval.
- Switch `url` to an AuraDB endpoint to move from local to production without changing any other code.
- Pass a custom `Neo4jConfig` to control volume paths, retry behaviour, and container naming in CI.